### Установка зависимостей

In [27]:
# https://microsoft.github.io/graphrag/get_started/
# https://developers.llamaindex.ai/python/framework/getting_started/starter_example_local/
%pip install -U raglite raglite[pandoc] lightrag-hku[api] scikit-learn ollama tqdm PyMuPDF numpy ipywidgets tqdm nest_asyncio pyvis matplotlib

  Using cached numpy-2.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.3 kB)
Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.61.1-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.man

### Подготовка функций для оценки.
Взято [отсюда](https://www.geeksforgeeks.org/nlp/evaluation-metrics-for-retrieval-augmented-generation-rag-systems/).

In [1]:
import numpy as np

# MRR
def mean_reciprocal_rank(y_true, y_pred):
    reciprocal_ranks = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        rr = 0
        for rank, doc in enumerate(pred_docs, start=1):
            if doc in true_docs:
                rr = 1 / rank
                break
        reciprocal_ranks.append(rr)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)
def ndcg(y_true, y_pred, k=5):
    ndcg_scores = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        pred_docs_k = pred_docs[:k]
        dcg = sum([1 / np.log2(idx + 2) if doc in true_docs else 0 for idx, doc in enumerate(pred_docs_k)])
        ideal_docs_k = true_docs[:k]
        idcg = sum([1 / np.log2(idx + 2) for idx, _ in enumerate(ideal_docs_k)])
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(ndcg_scores)
def recall_precision_at_k(y_true, y_pred, k=5):
    recall_list = []
    precision_list = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        top_k = pred_docs[:k]
        hits = len([doc for doc in top_k if doc in true_docs])
        recall_list.append(hits / len(true_docs) if true_docs else 0)
        precision_list.append(hits / k)
    return np.mean(recall_list), np.mean(precision_list)

# Example usage:
#y_true = [['doc1', 'doc2'], ['doc3']]
#y_pred = [['doc2', 'doc4'], ['doc5']]
#print("nDCG@5:", ndcg(y_true, y_pred))
#will print: nDCG@5: 0.3065735963827292

### Подготовка данных (фильтрация)
Для повышения качества оценки будут использованы данные, экспортированные в MarkDown формат (формулы в TeX-формате) через PaddleOCR (PPv3).
Данные представляют собой статьи по математике из открытого [источника](https://huggingface.co/datasets/PleIAs/Math-PDF/blob/main/math_pdf_tars/openalex_math_pdf_tar_31.tar). Большая часть статей в данном датасете позволяют использование экспорта текста без OCR.
Будет взято меньше 50 статей + 5 статей с релевантной темой для запросов.

Список доп. статей:
* https://math.berkeley.edu/~giventh/papers/qkf.pdf
* https://www.ams.org/journals/jams/2014-27-04/S0894-0347-2014-00797-9/S0894-0347-2014-00797-9.pdf
* https://www.cambridge.org/core/services/aop-cambridge-core/content/view/935C492E469B3B107B20F50DFE0C0F64/S2050509424001476a.pdf/a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
* https://personal.math.vt.edu/lmihalce/QKlectures(MSJ23).pdf
* и W1605366104.pdf из датасета

In [2]:
DATA_DIR="../../data"
RAW_DATA_DIR=f"{DATA_DIR}/openalex_math_pdf_tar_31"
PROCESSED_DATA_DIR=f"{DATA_DIR}/for_rag_2"
MATH_LIMIT=50
FIND_ALL_DOCS_POSTFIX=" Find all relevant documents."
COEF_K=5
MODEL_LLM="qwen3:8b"
MODEL_EMBED="embeddinggemma:300m"

In [45]:
from ollama import Client
qwen3_ollama = Client(
    host='http://localhost:11434'
)

def ollama_req_math(text, prompt_ask="Is this text about math?", model='qwen3:1.7b', text_limit=512):
    output_text = ""
    sys_prompt="You are a helpful assistant"
    prompt = f"{prompt_ask} The text to analyze is below:\n--\n{text[:text_limit]}\n--\nReturn only YES or NO answer without ANY explanation."
    for part in qwen3_ollama.generate(model, prompt=prompt, system=sys_prompt, stream=True):
        if part.thinking:
            continue
            print(part.thinking, end='', flush=True)
        output_text += part.response
    return output_text

In [38]:
from pathlib import Path
from tqdm.notebook import tqdm
import fitz
from pymupdf import FileDataError
import os
import shutil

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
for file in tqdm(list(Path(RAW_DATA_DIR).glob("*.pdf"))):
    text = ""
    try:
        with fitz.open(file) as doc:
            for page in doc:  # iterate the document pages
                text += page.get_text()
    except FileDataError as e:
        print(f"Error reading file {file.name}: {e}")
        continue
    res = ollama_req_math(text=text)
    #print(f"File: {file.name}, Result: {res}")
    if res == "YES":
        # just copy file
        print(f"Copying file: {file.name}")
        shutil.copy(file, os.path.join(PROCESSED_DATA_DIR, file.name))
        if len(math_files) >= MATH_LIMIT:
            print(f"Reached math file limit of {MATH_LIMIT}. Stopping.")
            break
    elif res == "NO":
        print(f"Skipping file: {file.name}")
    else:
        print(f"Unexpected result for file {file.name}: {res}")

    

  0%|          | 0/5000 [00:00<?, ?it/s]

Skipping file: W4295565825.pdf
Copying file: W2957174555_1.pdf
Copying file: W3167731107_2.pdf
Copying file: W2796609034.pdf
Skipping file: W4287119660_1.pdf
Copying file: W4393200428.pdf
Skipping file: W2602179841_3.pdf
Copying file: W4313001346.pdf
Skipping file: W2512409555_1.pdf
Skipping file: W4312320968.pdf
Skipping file: W4386767063.pdf
Copying file: W4377086487_5.pdf
MuPDF error: library error: FT_New_Memory_Face(UBPIRB+CMMI9): broken table

Copying file: W2465613768.pdf
Copying file: W4289128375.pdf
Copying file: W4226456969_2.pdf
Skipping file: W818768447.pdf
Copying file: W4317037187_2.pdf
Copying file: W4396577029.pdf
Copying file: W4283689522.pdf
Skipping file: W3188079605_3.pdf
Copying file: W2164376650_2.pdf
Copying file: W4246716229.pdf
Skipping file: W2791154508.pdf
Error reading file W4394623322.pdf: Failed to open file '../../data/openalex_math_pdf_tar_31/W4394623322.pdf'.
Copying file: W4300932590_1.pdf
Copying file: W2551158135.pdf
Skipping file: W2802454767_1.pdf


In [46]:
# Now second analyze with more params
for file in tqdm(list(Path(PROCESSED_DATA_DIR).glob("*.pdf"))):
    text = ""
    try:
        with fitz.open(file) as doc:
            for page in doc:  # iterate the document pages
                text += page.get_text()
    except FileDataError as e:
        print(f"Error reading file {file.name}: {e}")
        continue
    res = ollama_req_math(text=text, prompt_ask="Is this text about math and contains formulas?", model='qwen3:8b', text_limit=2048)
    #print(f"File: {file.name}, Result: {res}")
    if res == "YES":
        print(f"Leaving file: {file.name}")
    elif res == "NO":
        print(f"Removing file: {file.name}")
        os.remove(file)
    else:
        print(f"Unexpected result for file {file.name}: {res}")

  0%|          | 0/50 [00:00<?, ?it/s]

Leaving file: W2957174555_1.pdf
Removing file: W3167731107_2.pdf
Leaving file: W2796609034.pdf
Removing file: W4393200428.pdf
Leaving file: W4313001346.pdf
Leaving file: W4377086487_5.pdf
MuPDF error: library error: FT_New_Memory_Face(UBPIRB+CMMI9): broken table

Leaving file: W2465613768.pdf
Leaving file: W4289128375.pdf
Leaving file: W4226456969_2.pdf
Leaving file: W4317037187_2.pdf
Removing file: W4396577029.pdf
Leaving file: W4283689522.pdf
Removing file: W2164376650_2.pdf
Removing file: W4246716229.pdf
Removing file: W4300932590_1.pdf
Removing file: W2551158135.pdf
Removing file: W2223722977_3.pdf
Leaving file: W4206656803.pdf
Removing file: W3157468823.pdf
Leaving file: W4324126587_2.pdf
Removing file: W4297662594.pdf
Leaving file: W1603196302_1.pdf
Leaving file: W2158760784.pdf
Removing file: W2508541584_1.pdf
Leaving file: W2519367019.pdf
Leaving file: W2108242840.pdf
Leaving file: W2999061577_2.pdf
Leaving file: W25036734.pdf
Leaving file: W4285891493.pdf
Removing file: W30190

In [48]:
files_to_add = [
    "QKlectures(MSJ23).pdf",
    "qkf.pdf",
    "S0894-0347-2014-00797-9.pdf",
    "a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf"
]

OLD_DATA_DIR=f"{DATA_DIR}/for_rag"

for file_name in files_to_add:
    src_path = os.path.join(OLD_DATA_DIR, file_name)
    dst_path = os.path.join(PROCESSED_DATA_DIR, file_name)
    if os.path.exists(src_path):
        print(f"Adding file: {file_name}")
        shutil.copy(src_path, dst_path)
    else:
        raise FileNotFoundError(f"File to add not found: {file_name}")

Adding file: QKlectures(MSJ23).pdf
Adding file: qkf.pdf
Adding file: S0894-0347-2014-00797-9.pdf
Adding file: a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf


In [3]:
from pathlib import Path

math_files = [ file.name for file in list(Path(PROCESSED_DATA_DIR).glob("*.pdf"))]
print(f"Total math-related files prepared for RAG: {len(math_files)}")
for file in math_files:
    print(f"* {file}")

Total math-related files prepared for RAG: 35
* W2957174555_1.pdf
* W2796609034.pdf
* W4313001346.pdf
* W4377086487_5.pdf
* W2465613768.pdf
* W4289128375.pdf
* W4226456969_2.pdf
* W4317037187_2.pdf
* W4283689522.pdf
* W4206656803.pdf
* W4324126587_2.pdf
* W1603196302_1.pdf
* W2158760784.pdf
* W2519367019.pdf
* W2108242840.pdf
* W2999061577_2.pdf
* W25036734.pdf
* W4285891493.pdf
* W2945171940.pdf
* W2171382235.pdf
* W3118338062.pdf
* W4319655444_7.pdf
* W3211598426.pdf
* W3152618620_1.pdf
* W2963449700_2.pdf
* W4394719147.pdf
* W4310022274_2.pdf
* W4376956170.pdf
* W4386002119.pdf
* W2972953549_2.pdf
* W3128183679.pdf
* S0894-0347-2014-00797-9.pdf
* QKlectures(MSJ23).pdf
* a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
* qkf.pdf


### Тестирование

In [4]:
# Определяем ground-truth

# query: [docs]

ground_truth = {
    "Give information about K theory": [
        "W1605366104.pdf",
        "QKlectures(MSJ23).pdf",
        "qkf.pdf",
        "S0894-0347-2014-00797-9.pdf",
        "a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf",
    ],
    "Write proof of the Pieri-type formula": ["W1605366104.pdf", "QKlectures(MSJ23).pdf"],
    "What does this formula mean? `v(h) < v(i) < v(l)`": ["W1605366104.pdf"],
    "Show Forbidden subsequences in chains in the k-Bruhat order": ["W1605366104.pdf", "QKlectures(MSJ23).pdf"],
    "What is `∧i(S) · det(S∨) = ∧k−i(S∨)`": ["W1605366104.pdf"],
}

#### RAGLite

In [7]:
!ollama pull qwen3:8b
!ollama pull embeddinggemma:300m

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling a3de86cd1c13: 100% ▕██████████████████▏ 5.2 GB                         
pulling ae370d884f10: 100% ▕██████████████████▏ 1.7 KB                         
pulling d18a5cc71b84: 100% ▕██████████████████▏  11 KB                         
pulling cff3f395ef37: 100% ▕██████████████████▏  120 B                         
pulling 05a61d37b084: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 0800cbac9c20: 100% ▕██████████████████▏ 621 MB                         
pulling 1adbfec9dcf0: 100% ▕██████████████████▏ 8.4 KB                         
pulling 45dc10444b87: 100% ▕██████████████████▏   34 B                         
pulling 3901c6a1d7c2: 100% ▕████████████████

In [14]:
import os
from pathlib import Path
from raglite import RAGLiteConfig
from raglite import Document, insert_documents

# Set Ollama API base URL
os.environ["OLLAMA_API_BASE"] = f"http://localhost:11434"  # Ensure SERVER_HOST is defined

# Configure RAGLite
# Picked from here: https://github.com/superlinear-ai/raglite/issues/85
raglite_config = RAGLiteConfig(
    db_url="duckdb:///raglite.db",
    llm=f"ollama/{MODEL_LLM}",
    embedder=f"ollama/{MODEL_EMBED}"
#    chunk_max_size=300,  # Chinese vector models are generally recommended to set around 512 context size
)

In [2]:
from tqdm.notebook import tqdm

# took ~21m
raglite_documents = []
for file_name in tqdm(math_files):
    raglite_documents.append(Document.from_path(Path(os.path.join(PROCESSED_DATA_DIR, file_name))))
insert_documents(documents=raglite_documents, config=raglite_config)

NameError: name 'math_files' is not defined

In [17]:
from raglite import add_context, rag, retrieve_context, vector_search

from dataclasses import replace
my_config = replace(raglite_config, search_method=vector_search)  # Or `hybrid_search`, `search_and_rerank_chunks`, ...

def rag_ask(text: str, silent: bool = False):
    response = ""

    # Retrieve relevant chunk spans with the configured search method
    chunk_spans = retrieve_context(query=text, num_chunks=5, config=my_config)

    # Append a RAG instruction based on the user prompt and context to the message history
    messages = []  # Or start with an existing message history
    messages.append(add_context(user_prompt=text, context=chunk_spans))

    # Stream the RAG response and append it to the message history
    stream = rag(messages, config=my_config)
    for update in stream:
        if not silent:
            print(update, end="")
        response += update

    # Access the documents referenced in the RAG context
    documents = [chunk_span.document for chunk_span in chunk_spans]
    if not silent:
        for doc in documents:
            print(f"document: {doc}")

    return response, documents

In [18]:
test_res, test_docs = rag_ask("Give information about K theory. Find all relevant documents.")
for doc in test_docs:
    print(f"Relevant document: {doc.filename}")

руж
Okay, let's see. The user is asking for information about K theory, specifically to find all relevant documents from the provided context. First, I need to go through each of the three documents and identify the parts that discuss K theory.

Starting with the first document, which seems to be about quantum K theory. There's a mention of K(Gr(2,4)) and some equations involving line bundles. Then there's a section on positivity in K theory, referencing Buch and Brion. Theorems 2.10 and 2.12 are about K-theoretic positivity, and there's a proof sketch using the Kawamata-Viehweg vanishing theorem. Also, there are propositions about presentations of K theory rings for Grassmannians and flag varieties. 

The second document is more about general K theory, including definitions of rational singularities, theorems on positivity, and presentations of K theory rings. It also mentions the Schubert package, Schubert basis, dual basis, and Pieri-Chevalley formulas. There's a section on the K-th

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
raglite_mrr_all = []
raglite_ndcg_all = []
raglite_recall_all = []
raglite_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = rag_ask(req + FIND_ALL_DOCS_POSTFIX, silent=True)
    retrieved_doc_names = [doc.filename for doc in documents]
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    raglite_mrr_all.append(mrr)
    raglite_ndcg_all.append(ndcg_score)
    raglite_recall_all.append(recall)
    raglite_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall RAGLite nDCG@{COEF_K}: {np.mean(raglite_ndcg_all)}")
print(f"Overall RAGLite MRR: {np.mean(raglite_mrr_all)}")
print(f"Overall RAGLite recall@{COEF_K}: {np.mean(raglite_recall_all)}")
print(f"Overall RAGLite precision@{COEF_K}: {np.mean(raglite_precision_all)}")
# took ~1m 11s

  0%|          | 0/5 [00:00<?, ?it/s]

Request: Give information about K theory
Retrieved documents: ['QKlectures(MSJ23).pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'QKlectures(MSJ23).pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.8687949224876582
MRR: 1.0
recall@5, precision@5: 0.8, 0.8
-----
Request: Write proof of the Pieri-type formula
Retrieved documents: ['W3118338062.pdf', 'QKlectures(MSJ23).pdf', 'W2957174555_1.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf']
nDCG: 0.38685280723454163
MRR: 0.5
recall@5, precision@5: 0.5, 0.2
-----
Request: What does this formula mean? `v(h) < v(i) < v(l)`
Retrieved documents: ['W2957174555_1.pdf', 'W4317037187_2.pdf', 'W4377086487_5.pdf']
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Re

#### LightRAG

In [5]:
# From https://github.com/leovianaf/light-rag-tutorial/blob/main/notebooks/light_rag_example.ipynb
# and https://www.kaggle.com/code/parthsanghavi017/evaline-lightrag
import logging
from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from lightrag.llm.ollama import ollama_model_complete, ollama_embed
from functools import partial
import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(format="%(levelname)s:%(message)s", level=logging.INFO)

rag = LightRAG(
    working_dir="./lightrag_data",
    llm_model_func=ollama_model_complete,
    llm_model_name=MODEL_LLM,
    llm_model_kwargs={"host": "http://localhost:11434", "options": {"num_ctx": 32678}, "timeout": 300},
    llm_model_max_async=1,
    embedding_func_max_async=1,
    embedding_func=EmbeddingFunc(
        embedding_dim=768,
        max_token_size=8192,
        func=partial(
            ollama_embed.func,
            embed_model=MODEL_EMBED,
#            options={"num_thread": 2},
            host="http://localhost:11434"
        )
    ),
    default_embedding_timeout=180,
)

await rag.initialize_storages()

INFO: [] Loaded graph from ./lightrag_data/graph_chunk_entity_relation.graphml with 10 nodes, 0 edges
INFO:Load (10, 768) data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_entities.json'} 10 data
INFO:Load (0, 768) data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_relationships.json'} 0 data
INFO:Load (35, 768) data
INFO:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './lightrag_data/vdb_chunks.json'} 35 data
INFO: [] Process 115573 KV load full_docs with 35 records
INFO: [] Process 115573 KV load text_chunks with 35 records
INFO: [] Process 115573 KV load full_entities with 10 records
INFO: [] Process 115573 KV load full_relations with 0 records
INFO: [] Process 115573 KV load entity_chunks with 10 records
INFO: [] Process 115573 KV load relation_chunks with 0 records
INFO: [] Process 115573 KV load llm_response_cache with 82 records
INFO: [] Process 115573 doc status load doc_st

In [ ]:
# took ~8m
import os
from tqdm.notebook import tqdm
for file in tqdm(math_files):
    print(f"Adding document: {file}")
    rag.insert(os.path.join(PROCESSED_DATA_DIR, file))

  0%|          | 0/35 [00:00<?, ?it/s]

INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-c00d0aebe199de90f7b25e8fd64c719a
INFO: Embedding func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)


Adding document: W2957174555_1.pdf


INFO:  == LLM cache == saving: default:extract:81b59ffd7409c8c17a79c94c211bbccb
INFO:  == LLM cache == saving: default:extract:317185b1ef87c6ddba9ad23c2e562c2a
INFO: Chunk 1 of 1 extracted 0 Ent + 0 Rel chunk-c00d0aebe199de90f7b25e8fd64c719a
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 0 entities from doc-c00d0aebe199de90f7b25e8fd64c719a (async: 2)
INFO: Phase 2: Processing 0 relations from doc-c00d0aebe199de90f7b25e8fd64c719a (async: 2)
INFO: Phase 3: Updating final 0(0+0) entities and  0 relations from doc-c00d0aebe199de90f7b25e8fd64c719a
INFO: Completed merging: 0 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 0 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-ba605b58feb97d2d90f607b6a02f1525


Adding document: W2796609034.pdf


...

Adding document: a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf


INFO:  == LLM cache == saving: default:extract:95134b5125c03e54af1746b996a4e1ee
INFO:  == LLM cache == saving: default:extract:e97b308af78bac899331f29133c4ea4f
INFO: Chunk 1 of 1 extracted 0 Ent + 0 Rel chunk-4b2e55ac66f1968ddefab1b1167863a4
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 0 entities from doc-4b2e55ac66f1968ddefab1b1167863a4 (async: 2)
INFO: Phase 2: Processing 0 relations from doc-4b2e55ac66f1968ddefab1b1167863a4 (async: 2)
INFO: Phase 3: Updating final 0(0+0) entities and  0 relations from doc-4b2e55ac66f1968ddefab1b1167863a4
INFO: Completed merging: 0 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 10 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-7ab0f2a79b6fa2fe131e876ae4180c66


Adding document: qkf.pdf


INFO:  == LLM cache == saving: default:extract:01720bbd152888389ed08935cc05bc60
INFO:  == LLM cache == saving: default:extract:2a6bef01cb8962cd4a34b754c6ec3cd0
INFO: Chunk 1 of 1 extracted 0 Ent + 0 Rel chunk-7ab0f2a79b6fa2fe131e876ae4180c66
INFO: Merging stage 1/1: unknown_source
INFO: Phase 1: Processing 0 entities from doc-7ab0f2a79b6fa2fe131e876ae4180c66 (async: 2)
INFO: Phase 2: Processing 0 relations from doc-7ab0f2a79b6fa2fe131e876ae4180c66 (async: 2)
INFO: Phase 3: Updating final 0(0+0) entities and  0 relations from doc-7ab0f2a79b6fa2fe131e876ae4180c66
INFO: Completed merging: 0 entities, 0 extra entities, 0 relations
INFO: [] Writing graph with 10 nodes, 0 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 1/1: unknown_source
INFO: Enqueued document processing pipeline stopped


ERROR:Task was destroyed but it is pending!
task: <Task pending name='Task-3160' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-3161' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.12/asyncio/timeouts.py:97: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  async def __aexit__(
ERROR:Task was destroyed but it is pending!
task: <Task pending name='Task-3161' coro=<Kernel.shell_main() running at /home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]>


In [15]:
await rag.finalize_storages()

INFO: Successfully finalized 12 storages


In [29]:
rag.clear_cache()

AttributeError: 'function' object has no attribute 'clear_cache'

In [6]:
from lightrag.base import QueryParam
import re

resp = rag.query(
            "Give information about K theory. Find all relevant documents.",
            param=QueryParam(mode="local", stream=False),
        )
display(resp)
[ x.split("] ", 1)[1] for x in re.findall(r'- \[\d\] .+\.pdf', str(resp)) ]

INFO: Embedding func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO: Query nodes: K theory (top_k:40, cosine:0.2)
INFO: Local query: 8 entites, 0 relations
INFO: Raw search results: 8 entities, 0 relations, 0 vector chunks
INFO: After truncation: 8 entities, 0 relations
INFO: Selecting 8 from 8 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 8 -> 8 (deduplicated 0)
INFO: Final context: 8 entities, 0 relations, 8 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8
INFO:  == LLM cache == Query cache hit, using cached response as query result


'### Information about K Theory and Relevant Documents\n\nK theory is a branch of mathematics that studies vector bundles and their invariants, with applications in algebraic geometry, topology, and functional analysis. While the provided context does not explicitly define K theory or its mathematical foundations, it includes several documents that may contain relevant information or references to the topic. Below is an analysis of the relevant documents identified from the **Knowledge Graph Data** and **Document Chunks**:\n\n---\n\n#### **Relevant Documents Related to K Theory**\n\n1. **S0894-0347-2014-00797-9.pdf**  \n   - This document is described as a research paper with scholarly content. While its title is not explicitly provided, the journal identifier (`S0894-0347`) suggests it may belong to a mathematics or physics journal. Research papers in these fields often discuss advanced topics like K theory, particularly in contexts such as algebraic K-theory or topological K-theory. 

['QKlectures(MSJ23).pdf',
 'S0894-0347-2014-00797-9.pdf',
 'W2519367019.pdf',
 'W4377086487_5.pdf',
 'W2171382235.pdf']

In [ ]:
def lightrag_ask(prompt: str, silent=False):
    resp = rag.query(prompt, param=QueryParam(mode="hybrid", stream=False))
    if not silent:
        print(resp)
    return resp, [ x.split("] ", 1)[1] for x in re.findall(r'- \[\d\] .+\.pdf', str(resp)) ]

In [12]:
lightrag_ask("Give information about K theory Find all relevant documents.", silent=True)

INFO: LLM func: 1 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: local:keywords:55f0f78a569886fc3c1e8f46ba648678
INFO: Query nodes: K theory (top_k:40, cosine:0.2)
INFO: Local query: 8 entites, 0 relations
INFO: Raw search results: 8 entities, 0 relations, 0 vector chunks
INFO: After truncation: 8 entities, 0 relations
INFO: Selecting 8 from 8 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 8 -> 8 (deduplicated 0)
INFO: Final context: 8 entities, 0 relations, 8 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8
INFO:  == LLM cache == saving: local:query:cbdc942da7964a3e2774ef7840b08817


('The provided context does not explicitly mention **K-theory** or any direct references to its concepts, applications, or related documents. However, the following entries in the **Knowledge Graph Data** and **Document Chunks** may be relevant to mathematical or theoretical research, as they involve PDF files with unspecified content:\n\n---\n\n### Relevant Documents (Potential Candidates)\n1. **S0894-0347-2014-00797-9.pdf**  \n   - A research paper document described as containing scholarly content. The journal identifier `S0894-0347` suggests it may belong to a mathematics or theoretical physics journal, which could plausibly include topics like K-theory. However, the absence of explicit metadata prevents confirmation.\n\n2. **QKlectures(MSJ23).pdf**  \n   - A lecture material file related to the "MSJ23" project. While the project name is unspecified, lecture notes often cover advanced mathematical topics, including K-theory. Further inspection of the document would be required to v

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

lightrag_mrr_all = []
lightrag_ndcg_all = []
lightrag_recall_all = []
lightrag_precision_all = []
for req, true_docs in tqdm(ground_truth.items()):
    print(f"Request: {req}")
    response, documents = lightrag_ask(req + FIND_ALL_DOCS_POSTFIX, silent=True)
    retrieved_doc_names = documents
    print(f"Retrieved documents: {retrieved_doc_names}")
    print(f"Ground truth documents: {true_docs}")

    mrr = mean_reciprocal_rank([true_docs], [retrieved_doc_names])
    ndcg_score = ndcg([true_docs], [retrieved_doc_names], k=COEF_K)
    recall, precision = recall_precision_at_k([true_docs], [retrieved_doc_names], k=COEF_K)

    lightrag_mrr_all.append(mrr)
    lightrag_ndcg_all.append(ndcg_score)
    lightrag_recall_all.append(recall)
    lightrag_precision_all.append(precision)
    print(f"nDCG: {ndcg_score}\nMRR: {mrr}\nrecall@{COEF_K}, precision@{COEF_K}: {recall}, {precision}")
    print("-----")
print(f"Overall LightRAG nDCG@{COEF_K}: {np.mean(lightrag_ndcg_all)}")
print(f"Overall LightRAG MRR: {np.mean(lightrag_mrr_all)}")
print(f"Overall LightRAG recall@{COEF_K}: {np.mean(lightrag_recall_all)}")
print(f"Overall LightRAG precision@{COEF_K}: {np.mean(lightrag_precision_all)}")
# took ~1 24s + cached (38)

  0%|          | 0/5 [00:00<?, ?it/s]

INFO: Query nodes: K theory (top_k:40, cosine:0.2)


Request: Give information about K theory


INFO: Local query: 8 entites, 0 relations
INFO: Raw search results: 8 entities, 0 relations, 0 vector chunks
INFO: After truncation: 8 entities, 0 relations
INFO: Selecting 8 from 8 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 8 -> 8 (deduplicated 0)
INFO: Final context: 8 entities, 0 relations, 8 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6 E1/7 E1/8
INFO:  == LLM cache == Query cache hit, using cached response as query result


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'QKlectures(MSJ23).pdf', 'W2519367019.pdf', 'W4377086487_5.pdf', 'W2171382235.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf', 'qkf.pdf', 'S0894-0347-2014-00797-9.pdf', 'a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf']
nDCG: 0.5531464700081437
MRR: 1.0
recall@5, precision@5: 0.4, 0.4
-----
Request: Write proof of the Pieri-type formula


...

Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'QKlectures(MSJ23).pdf', 'W4377086487_5.pdf', 'W2171382235.pdf', 'W4317037187_2.pdf']
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Request: Show Forbidden subsequences in chains in the k-Bruhat order


INFO:  == LLM cache == saving: local:keywords:557dc8b1b6f7c3ee95ed3c24c2b7c270
INFO: Query nodes: k-Bruhat order, chains, subsequences (top_k:40, cosine:0.2)
INFO: Local query: 6 entites, 0 relations
INFO: Raw search results: 6 entities, 0 relations, 0 vector chunks
INFO: After truncation: 6 entities, 0 relations
INFO: Selecting 6 from 6 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 6 -> 6 (deduplicated 0)
INFO: Final context: 6 entities, 0 relations, 6 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5 E1/6
INFO:  == LLM cache == saving: local:query:589601d41b3aafa4728af8bc7e2b2b18


Retrieved documents: ['QKlectures(MSJ23).pdf', 'S0894-0347-2014-00797-9.pdf', 'W2519367019.pdf', 'W2796609034.pdf', 'W2465613768.pdf']
Ground truth documents: ['W1605366104.pdf', 'QKlectures(MSJ23).pdf']
nDCG: 0.6131471927654584
MRR: 1.0
recall@5, precision@5: 0.5, 0.2
-----
Request: What is `∧i(S) · det(S∨) = ∧k−i(S∨)`


INFO:  == LLM cache == saving: local:keywords:55a75ba4e113e3f87a7a28e74ba0e048
INFO: Query nodes: ∧i(S), det(S∨), ∧k−i(S∨), Logical operators, Determinant function, Variables i, k, S (top_k:40, cosine:0.2)
INFO: Local query: 5 entites, 0 relations
INFO: Raw search results: 5 entities, 0 relations, 0 vector chunks
INFO: After truncation: 5 entities, 0 relations
INFO: Selecting 5 from 5 entity-related chunks by vector similarity
INFO: Round-robin merged chunks: 5 -> 5 (deduplicated 0)
INFO: Final context: 5 entities, 0 relations, 5 chunks
INFO: Final chunks S+F/O: E1/1 E1/2 E1/3 E1/4 E1/5
INFO:  == LLM cache == saving: local:query:eaf72f948034323ddb6d4517b4e7053b


Retrieved documents: ['S0894-0347-2014-00797-9.pdf', 'QKlectures(MSJ23).pdf', 'W4377086487_5.pdf', 'W2519367019.pdf', 'W4317037187_2.pdf']
Ground truth documents: ['W1605366104.pdf']
nDCG: 0.0
MRR: 0.0
recall@5, precision@5: 0.0, 0.0
-----
Overall LightRAG nDCG@5: 0.31062929400162875
Overall LightRAG MRR: 0.5
Overall LightRAG recall@5: 0.27999999999999997
Overall LightRAG precision@5: 0.16
